# Video Understanding — Temporal Modeling Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Frame sampler

Uniform and dense samplers that work on a list of frames (or a video tensor).

In [ ]:
```python

import numpy as np

def sample_uniform(num_frames_total, T):

    if num_frames_total <= T:

        return list(range(num_frames_total)) + [num_frames_total - 1] * (T - num_frames_total)

    step = num_frames_total / T

    return [int(i * step) for i in range(T)]

def sample_dense(num_frames_total, T, rng=None):

    rng = rng or np.random.default_rng()

    if num_frames_total <= T:

        return list(range(num_frames_total)) + [num_frames_total - 1] * (T - num_frames_total)

    start = int(rng.integers(0, num_frames_total - T + 1))

    return list(range(start, start + T))

In [ ]:
```

Both return `T` indices that you use to slice the video tensor.

### Step 2: A 2D+pool baseline

Run a 2D ResNet-18 over every frame, average-pool features, classify.

In [ ]:
```python

import torch

import torch.nn as nn

from torchvision.models import resnet18, ResNet18_Weights

class FramePool(nn.Module):

    def __init__(self, num_classes=400, pretrained=True):

        super().__init__()

        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None

        backbone = resnet18(weights=weights)

        self.features = nn.Sequential(*(list(backbone.children())[:-1]))  # global avg pool kept

        self.head = nn.Linear(512, num_classes)

    def forward(self, x):

        # x: (N, T, 3, H, W)

        N, T = x.shape[:2]

        x = x.view(N * T, *x.shape[2:])

        feats = self.features(x).view(N, T, -1)

        pooled = feats.mean(dim=1)

        return self.head(pooled)

model = FramePool(num_classes=10)

x = torch.randn(2, 8, 3, 224, 224)

print(f"output: {model(x).shape}")

print(f"params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
```

Eleven million parameters, ImageNet pretrained, runs per-frame, averages, classifies. This baseline is often within 5-10 points of proper 3D models on appearance-heavy tasks — sometimes better, because it reuses a stronger ImageNet backbone.

### Step 3: An I3D-style inflated 3D conv

Turn a single 2D conv into a 3D conv by repeating weights along a new time axis.

In [ ]:
```python

def inflate_2d_to_3d(conv2d, time_kernel=3):

    out_c, in_c, kh, kw = conv2d.weight.shape

    weight_3d = conv2d.weight.data.unsqueeze(2)  # (out, in, 1, kh, kw)

    weight_3d = weight_3d.repeat(1, 1, time_kernel, 1, 1) / time_kernel

    conv3d = nn.Conv3d(in_c, out_c, kernel_size=(time_kernel, kh, kw),

                        padding=(time_kernel // 2, conv2d.padding[0], conv2d.padding[1]),

                        stride=(1, conv2d.stride[0], conv2d.stride[1]),

                        bias=False)

    conv3d.weight.data = weight_3d

    return conv3d

conv2d = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)

conv3d = inflate_2d_to_3d(conv2d, time_kernel=3)

print(f"2D weight shape:  {tuple(conv2d.weight.shape)}")

print(f"3D weight shape:  {tuple(conv3d.weight.shape)}")

x = torch.randn(1, 3, 8, 56, 56)

print(f"3D output shape:  {tuple(conv3d(x).shape)}")

In [ ]:
```

The division by `time_kernel` keeps the activation magnitudes roughly constant — important for not breaking batch-norm statistics on the first pass.

### Step 4: Factorised (2+1)D conv

Split a 3D conv into a 2D (spatial) and a 1D (temporal) conv. Same receptive field, fewer parameters, better accuracy on some benchmarks.

In [ ]:
```python

class Conv2Plus1D(nn.Module):

    def __init__(self, in_c, out_c, kernel_size=3):

        super().__init__()

        mid_c = (in_c * out_c * kernel_size * kernel_size * kernel_size) \

                // (in_c * kernel_size * kernel_size + out_c * kernel_size)

        self.spatial = nn.Conv3d(in_c, mid_c, kernel_size=(1, kernel_size, kernel_size),

                                 padding=(0, kernel_size // 2, kernel_size // 2), bias=False)

        self.bn = nn.BatchNorm3d(mid_c)

        self.act = nn.ReLU(inplace=True)

        self.temporal = nn.Conv3d(mid_c, out_c, kernel_size=(kernel_size, 1, 1),

                                  padding=(kernel_size // 2, 0, 0), bias=False)

    def forward(self, x):

        return self.temporal(self.act(self.bn(self.spatial(x))))

c = Conv2Plus1D(3, 64)

x = torch.randn(1, 3, 8, 56, 56)

print(f"(2+1)D output: {tuple(c(x).shape)}")

In [ ]:
```

A full R(2+1)D network is the same as a ResNet-18 with every 3x3 conv replaced by `Conv2Plus1D`.

## Exercises

In [ ]:
1. **(Easy)** Compute FLOPs (approximate) for FramePool with T=8 vs an I3D-style 3D ResNet with T=8. Justify why 2D+pool is 3-5x cheaper.
2. **(Medium)** Generate a synthetic video dataset: random balls moving in random directions, labelled by direction of motion ("left-to-right", "right-to-left", "diagonal-up"). Train FramePool on it. Show that it achieves near-chance accuracy, proving appearance alone is insufficient for motion tasks.
3. **(Hard)** Build an R(2+1)D-18 by replacing every Conv2d in a ResNet-18 with `Conv2Plus1D`. Inflate the first conv's weights from an ImageNet-pretrained ResNet-18. Train on the motion dataset from exercise 2 and beat FramePool.